<a href="https://colab.research.google.com/github/briskicedteaa/Filler-Name/blob/main/FOXP2_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **FOXP2 Evolution Project**

In this notebook, we are attempting to answer one main question: How FOXP2 changed across primate and human evolution, and which changes are predicted to affect the protein's function.

FOXP2 is a transcription factor involved in the development and function of neural systems associated with speech, language, and motor learning.

In this notebook, we will compare FOXP2 protein sequences across species to identify how the protein has changed throughout evolution.

The general workflow is:

1. Collecting FOXP2 protein sequences from different species

2. Filteribg the dataset for valid, sufficiently complete proteins

3. Aligning the sequences

4. Identifying amino acid differences

5. Reconstructing the evolutionary history of those differences

6. Analyzing whether particular changes may affect FOXP2 function

## Methods and Tools

Before starting our data preparation, we used several bioinformatics tools to collect, process, and compare FOXP2 protein sequences.

### UniProt

We used **UniProt** to collect FOXP2 protein sequences from different species. UniProt provides protein sequences along with identifiers and descriptions, which allowed us to build our initial FOXP2 dataset and filter out records that were not suitable for our analysis.

### Biopython

We used **Biopython** to work with protein sequence and alignment data in Python. We used it to read our sequence files, process the Clustal Omega alignment, compare each aligned protein with human FOXP2, and identify positions where the amino-acid sequences differed.

### Clustal Omega (EMBL-EBI)

We used **Clustal Omega**, provided through EMBL-EBI, to perform a multiple sequence alignment (MSA) of our FOXP2 proteins.

An MSA lines up corresponding regions of protein sequences so that we can compare the same positions across different species. This allowed us to observe conserved regions, substitutions, and insertions/deletions relative to human FOXP2.

Together, these tools allowed us to go from a collection of FOXP2 sequences to an aligned dataset where evolutionary differences between the proteins could be identified and analyzed.

In [48]:
#@markdown Main installs for parsing and extracting data.

!pip install BioPython
from Bio import SeqIO

In [ ]:
import gzip

with gzip.open("uniprotkb_FOXP2_2026_08_15.fasta.gz", "rt") as handle:
  records = list(SeqIO.parse(handle, "fasta"))

In [ ]:
#@markdown Printed Details For First Two Proteins

print(records[0])
print(records[1])

ID: sp|O00560|SDCB1_HUMAN
Name: sp|O00560|SDCB1_HUMAN
Description: sp|O00560|SDCB1_HUMAN Syntenin-1 OS=Homo sapiens OX=9606 GN=SDCBP PE=1 SV=1
Number of features: 0
Seq('MSLYPSLEDLKVDKVIQAQTAFSANPANPAILSEASAPIPHDGNLYPRLYPELS...PEV')
ID: sp|O15409|FOXP2_HUMAN
Name: sp|O15409|FOXP2_HUMAN
Description: sp|O15409|FOXP2_HUMAN Forkhead box protein P2 OS=Homo sapiens OX=9606 GN=FOXP2 PE=1 SV=2
Number of features: 0
Seq('MMQESATETISNSSMNQNGMSTLSSQLDAGSRDGRSSGDTSSEVSTVELLHLQQ...DLE')


In [51]:
#@markdown The dataset can have duplicates, incomplete protein sequences (fragments), or contain proteins that are not FOXP2. This cell creates df, adds columns, filters "Description" for FOXP2, drops proteins containing "fragment" in description, and removes duplicates.

import pandas as pd

# Created df and added columns

df = pd.DataFrame({
    "ID": [record.id for record in records],
    "Description": [record.description for record in records],
    "Sequence": [str(record.seq) for record in records],
    "Length": [len(record.seq) for record in records],
})

# Filtering for only FOXP2 in description column

foxp2_df = df[
    df["Description"].str.contains(
        "Forkhead box protein P2",
        case=False,
        na=False
    )
].copy()

# Removed proteins containing "Fragment" in description

foxp2_df = foxp2_df[
    ~foxp2_df["Description"].str.contains(
        "Fragment",
        case=False,
        na=False
    )
].copy()

# Dropped duplicates

foxp2_df = foxp2_df.drop_duplicates(
    subset="Sequence",
    keep="first"

).copy()

print("Remaining Proteins:", len(foxp2_df))
print("Total Proteins:", len(df))
print("Duplicate sequences:", len(foxp2_df) - foxp2_df["Sequence"].nunique())
df.head()

Remaining Proteins: 525
Total Proteins: 1227
Duplicate sequences: 0


,ID,Description,Sequence,Length
0,sp|O00560|SDCB1_HUMAN,sp|O00560|SDCB1_HUMAN Syntenin-1 OS=Homo sapie...,MSLYPSLEDLKVDKVIQAQTAFSANPANPAILSEASAPIPHDGNLY...,298
1,sp|O15409|FOXP2_HUMAN,sp|O15409|FOXP2_HUMAN Forkhead box protein P2 ...,MMQESATETISNSSMNQNGMSTLSSQLDAGSRDGRSSGDTSSEVST...,715
2,sp|O88712|CTBP1_MOUSE,sp|O88712|CTBP1_MOUSE C-terminal-binding prote...,MGSSHLLNKGLPLGVRPPIMNGPMHPRPLVALLDGRDCTVEMPILK...,441
3,sp|P0CF24|FOXP2_RAT,sp|P0CF24|FOXP2_RAT Forkhead box protein P2 OS...,MMQESATETISNSSMNQNGMSTLSSQLDAGSRDGRSSGDTSSEVST...,710
4,sp|P24863|CCNC_HUMAN,sp|P24863|CCNC_HUMAN Cyclin-C OS=Homo sapiens ...,MAGNFWQSSHYLQWILDKQDLLKERQKDLKFLSEEEYWKLQIFFTN...,283


In [ ]:
from google.colab import files

foxp2_df.to_csv('FOXP2-Proteins.csv', index = False)
files.download('FOXP2-Proteins.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
#@markdown We wanted to align the sequences from our csv using EMBL's Clustal Omega, so we made the file readable for the MSA tool by creating a fasta file containing the remaining proteins.

with open("FOXP2.fasta", "w") as f:
    for _, row in foxp2_df.iterrows():
        f.write(f">{row['ID']}\n")
        f.write(f"{row['Sequence']}\n")

files.download("FOXP2.fasta")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
#@markdown After briefly analyzing our dataset, we realized some incomplete proteins were not labeled as fragments. Sequences shorter than ~80% of our 715-amino-acid human FOXP2 reference were excluded because their large amounts of missing sequence could introduce misleading gaps into MSA.

foxp2_df[
    ["ID", "Description", "Length"]
].sort_values("Length").head(30)

# Dropped proteins below 570 AA

foxp2_df = foxp2_df[
    foxp2_df["Length"] >= 570
].copy()

print("Proteins remaining:", len(foxp2_df))
print("Below 570 aa:", (foxp2_df["Length"] < 570).sum())
print("570 aa or longer:", (foxp2_df["Length"] >= 570).sum())
foxp2_df["Length"].describe()

Proteins remaining: 494
Below 570 aa: 0
570 aa or longer: 494


,Length
count,494.000000
mean,701.016194
std,40.750761
min,590.000000
25%,684.250000
50%,710.000000
75%,729.750000
max,880.000000


In [ ]:
foxp2_df.to_csv("FinalFOXP2.csv", index=False)
files.download("FinalFOXP2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
#@markdown Prepared file for MSA

with open("FOXP2.fasta", "w") as f:
    for _, row in foxp2_df.iterrows():
        f.write(f">{row['ID']}\n")
        f.write(f"{row['Sequence']}\n")

files.download("FOXP2.fasta")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
#@markdown Next, we wanted to compare every aligned FOXP2 sequence against human FOXP2, record every position where they differ, and keep both alignment coordinates and human residue coordinates so the differences can later be mappes to domains, structure, or known variants.

# Importing BioPythons alignment reader

from Bio import AlignIO

# Loaded clustal omega alignment

alignment = AlignIO.read(
    "clustalo-I20260815-200247-0706-9045984-p1m.aln-clustal_num",
    "clustal"
)

# Found Human FOXP2 sequence in the aligment usung its UniProt accession (O15409)
human = next(record for record in alignment if "O15409" in record.id)

# To store every difference found
mutations = []

# To keep track of the actual amino acid number in hunan FOXP2
human_position = 0


# Comparing every aligned position

for i, human_aa in enumerate(human.seq):

  if human_aa != "-":
    human_position += 1

# Comparing every other protein to our reference (Human FOXP2)
  for record in alignment:

    if record.id == human.id:
      continue

    other_aa = record.seq[i]

    if human_aa == other_aa:
      continue

    if human_aa == "-":
      position = None
    else:
      position = human_position

    mutations.append({
        "ID": record.id,
        "Aligned Position": i + 1,
        "Human Position": position,
        "Human AA": human_aa,
        "Other AA": other_aa,
        "Gap": human_aa == "-" or other_aa == "-"
    })

# Mutation Table
mutation_df = pd.DataFrame(mutations)

print("Substitutions:", (~mutation_df["Gap"]).sum())
print("Gaps/indels:", mutation_df["Gap"].sum())
print("Total Mutation Rows:", len(mutation_df))
mutation_df.head()

Substitutions: 11157
Gaps/indels: 31026
Total Mutation Rows: 42183


,ID,Aligned Position,Human Position,Human AA,Other AA,Gap
0,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,1,NaN,-,M,True
1,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,2,NaN,-,S,True
2,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,3,NaN,-,V,True
3,tr|A0ACF8DDW5|A0ACF8DDW5_ARAGI,4,NaN,-,Q,True
4,tr|A0A6P7NA03|A0A6P7NA03_BETSP,5,NaN,-,M,True


In [40]:
#@markdown Splitting our mutation table into two separate datasets, then naming the substitutions.


# Separating gaps/indels
substitution_df = mutation_df[
    ~mutation_df["Gap"]
].copy()

indel_df = mutation_df[
    mutation_df["Gap"]
].copy()

# Creating mutation names
substitution_df["Mutation"] = (
    substitution_df["Human AA"].astype(str)
    + substitution_df["Human Position"].astype(int).astype(str)
    + substitution_df["Other AA"].astype(str)
)

# Counting
print("Substitutions:", len(substitution_df))
print("Gap-containing differences:", len(indel_df))
print(
    "Unique substitutions:",
    substitution_df["Mutation"].nunique()
)

# Displaying most common substitutions
substitution_df["Mutation"].value_counts().head(20)

Substitutions: 11157
Gap-containing differences: 31026
Unique substitutions: 1781


,count
Mutation,
N303T,441
S325N,371
S235N,167
A249S,126
S78G,90
Q189P,89
Q191P,89
Q188P,83
S42T,81


In [46]:
#@markdown Downloading files

mutation_df.to_csv("FOXP2_mutations.csv", index=False)
substitution_df.to_csv("FOXP2_substitutions.csv", index=False)
indel_df.to_csv("FOXP2_indels.csv", index=False)

files.download("FOXP2_mutations.csv")
files.download("FOXP2_substitutions.csv")
files.download("FOXP2_indels.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
#@markdown We wanted to make a separate dataset containing only unique amino-acid substitutions. Some substitutions appeared in many different FOXP2 proteins, so keeping every repeated observation would make later mutation-level analyses redundant.

#@markdown By removing duplicate mutation entries, we can see each distinct substitution only once while still keeping its human position and amino-acid change. This gives us a cleaner dataset for later comparisons, such as conservation, chemical differences between amino acids, structural location, and predicted functional effects.

unique_substitutions = (
    substitution_df[
        ["Mutation", "Human Position", "Human AA", "Other AA"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_substitutions.to_csv(
    "FOXP2_unique_substitutions.csv",
    index=False
)

print("Unique substitutions:", len(unique_substitutions))

Unique substitutions: 1781
